# 03 · NumPy 배열과 계산

Python 목록에서 NumPy 배열로 넘어가며 인덱싱, 배열 생성, 모양, 축, 통계를 비교합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 배열 만들기

**코드 → 코드 개념**: `np.array`는 기존 값을, `arange`는 간격 기준, `linspace`는 개수 기준으로 만든다.

**코드 사용법**: 배열의 값·모양·자료형을 확인한다.

In [ ]:
import numpy as np
a = np.array([2.0, 3.0, 4.0])
print(a, a.shape, a.ndim, a.size, a.dtype)
print(np.arange(0, 10, 2))
print(np.linspace(0, 8, 5))
print(np.zeros((2, 3)), np.ones((2, 2)))

**같은 결과를 얻는 방법과 선택 이유**

- 정해진 간격이면 `arange`, 시작과 끝을 포함해 정해진 점 개수가 필요하면 `linspace`가 명확하다. 부동소수 간격은 `arange`의 끝점이 예상과 달라질 수 있다.
- 빈 데이터 구조를 준비할 때 `zeros/ones`를 쓴다.

## 리스트와 배열 연산

**코드 → 코드 개념**: NumPy 산술은 원소별로 적용된다. Python 리스트의 `+`는 연결이다.

**코드 사용법**: 같은 두 배 값을 반복문·컴프리헨션·배열로 만든다.

In [ ]:
values = [2, 3, 4]
loop = []
for x in values:
    loop.append(x * 2)
comp = [x * 2 for x in values]
array = (np.array(values) * 2).tolist()
print(loop, comp, array)
assert loop == comp == array

**같은 결과를 얻는 방법과 선택 이유**

- 소규모 일반 객체는 반복문이 유연하다. 숫자 대량 계산은 NumPy 벡터 연산이 간결하다.
- `values * 2`는 `[2,3,4,2,3,4]`가 되므로 수치 계산과 혼동하지 않는다.

## 인덱싱과 슬라이싱

**코드 → 코드 개념**: `a[start:stop]`은 끝 위치 직전까지, 2차원 배열은 `[행, 열]`로 선택한다.

**코드 사용법**: 센서 표의 첫 열과 마지막 두 행을 선택한다.

In [ ]:
sensor = np.array([[70, 2.1], [75, 2.4], [82, 3.8]])
print(sensor[0, 1], sensor[:, 0], sensor[-2:, :])
print(sensor.reshape(1, 6).shape)

**같은 결과를 얻는 방법과 선택 이유**

- `sensor[:, 0]`은 1차원 결과, `sensor[:, 0:1]`은 2차원 결과다. 후속 함수가 요구하는 모양에 따라 고른다.
- `reshape`는 원소 수가 같을 때만 가능하며 데이터의 뜻까지 바꾸지는 않는다.

## 불리언 선택

**코드 → 코드 개념**: 비교 연산은 참·거짓 마스크를 만들고, 마스크로 행을 선택한다.

**코드 사용법**: 온도가 75 이상인 행을 찾는다.

In [ ]:
mask = sensor[:, 0] >= 75
print(mask)
print(sensor[mask])
print(sensor[(sensor[:, 0] >= 75) & (sensor[:, 1] < 3)])

**같은 결과를 얻는 방법과 선택 이유**

- NumPy에서 조건 결합은 `&`, `|`를 쓰고 각 비교를 괄호로 묶는다. Python의 `and/or`는 배열 전체의 진릿값을 결정할 수 없다.
- 표에 열 이름이 있으면 Pandas의 `df.loc[...]`가 의미를 읽기 쉽다.

## 축과 통계

**코드 → 코드 개념**: `axis=0`은 행을 모아 열별 값, `axis=1`은 열을 모아 행별 값을 만든다.

**코드 사용법**: 열별 평균·최댓값과 결측을 무시한 평균을 구한다.

In [ ]:
print(sensor.mean(axis=0), sensor.max(axis=0))
with_missing = np.array([2.1, np.nan, 3.8])
print(np.mean(with_missing), np.nanmean(with_missing))
print(np.std([1, 2, 3], ddof=0), np.std([1, 2, 3], ddof=1))

**같은 결과를 얻는 방법과 선택 이유**

- `np.mean`은 결측이 있으면 `nan`을 낸다. `np.nanmean`은 결측을 제외한다. 제외된 건수를 함께 기록한다.
- `ddof=0`은 모집단 표준편차, `ddof=1`은 표본 표준편차다. Pandas `.std()` 기본값은 `ddof=1`이므로 비교 시 맞춘다.

## 브로드캐스팅과 표준화

**코드 → 코드 개념**: 모양이 맞으면 작은 배열이 큰 배열의 각 행에 적용된다.

**코드 사용법**: 서로 다른 단위의 센서 열을 열별로 표준화한다.

In [ ]:
mean = sensor.mean(axis=0)
std = sensor.std(axis=0)
z = (sensor - mean) / std
print(z.round(2))
print(np.where(sensor[:, 1] >= 3.0, "watch", "normal"))

**같은 결과를 얻는 방법과 선택 이유**

- `np.where`는 배열 전체를 두 결과로 나눌 때 좋다. 복잡한 여러 단계 조건은 `np.select`나 Pandas 분류 열이 읽기 쉽다.
- 표준편차가 0인 열은 나눌 수 없다. 실제 분석에서는 상수 열을 먼저 확인한다.

## 원본 학습 자료

[`1. lecture/01_Numpy`](../1.%20lecture/01_Numpy), [`4. Summary/06_Numpy/Numpy.md`](../4.%20Summary/06_Numpy/Numpy.md)